# MAREA and NIDEM against the number of scenes: the four sites

Reads the per-site results of `03_marea_vs_nidem_scene_count_site.ipynb`
(`results/scene_count_curves_nb/<site>.csv`, one row per draw and method) and draws them
together: per site, one MAREA curve and one NIDEM curve of the centred RMSE against the number
of scenes, and the bias below. Each point is the median over the draws that have a score; the
band spans 10–90 % of them.

## Parameters

In [ ]:
import os

SITES   = ["villaviciosa", "escalda", "wadden", "ems"]
NB_DIR  = os.environ.get("NB_OUT", "results/scene_count_curves_nb")
NAMES   = {"villaviciosa": "Ría de Villaviciosa", "escalda": "Westerschelde",
           "wadden": "Wadden Sea (Vlie)", "ems": "Ems-Dollard"}
TRUTH   = {"villaviciosa": "194 RTK points", "escalda": "Vaklodingen, 100 000 px",
           "wadden": "Vaklodingen, 100 000 px", "ems": "Vaklodingen, 100 000 px"}
STYLE   = {"MAREA": dict(color="#0A7FA8", marker="s"), "NIDEM": dict(color="#B7791F", marker="o")}

## 0 · Setup

In [ ]:
for _ in range(3):                                # this notebook lives in notebooks/validation/
    if os.path.isdir("pyintertidal"):
        break
    os.chdir("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

## 1 · The per-site results

A site without its CSV is left out, and the cell says so.

In [ ]:
frames = []
for site in SITES:
    path = f"{NB_DIR}/{site}.csv"
    if os.path.exists(path):
        frames.append(pd.read_csv(path))
    else:
        print(f"{site}: no {path} yet, left out")
df = pd.concat(frames, ignore_index=True)
sites = [s for s in SITES if s in set(df.site)]

table = df.pivot_table(index=["site", "N"], columns="method", values="rmse_centred", aggfunc="median")
table["NIDEM computable"] = (df[df.method == "NIDEM"].assign(ok=lambda d: d.status == "ok")
                             .groupby(["site", "N"]).ok.sum().astype(int))
table["draws"] = df[df.method == "NIDEM"].groupby(["site", "N"]).size()
table.reindex(sites, level=0).round(3)

## 2 · The four-site figure

Top row: centred RMSE against the number of scenes, MAREA one curve and NIDEM another. Bottom
row: the bias, median(product − truth). The N axis is logarithmic; the last point of each curve
is the whole pool.

In [ ]:
fig, axes = plt.subplots(2, len(sites), figsize=(4.6 * len(sites), 8), sharex="col", squeeze=False)
for k, site in enumerate(sites):
    for method, st in STYLE.items():
        g = df[(df.site == site) & (df.method == method)].groupby("N")
        for ax, col in zip(axes[:, k], ("rmse_centred", "bias")):
            med = g[col].median().dropna()
            ax.fill_between(med.index, g[col].quantile(0.1).loc[med.index], g[col].quantile(0.9).loc[med.index],
                            color=st["color"], alpha=0.15, lw=0)
            ax.plot(med.index, med.values, "-", lw=2, ms=5, label=method, **st)
    axes[0, k].set_title(f"{NAMES[site]}\n{TRUTH[site]}", loc="left", fontsize=11)
    axes[1, k].axhline(0, color="0.5", lw=0.8)
    for ax in axes[:, k]:
        ax.set_xscale("log"); ax.set_xticks(sorted(df[df.site == site].N.unique()))
        ax.xaxis.set_major_formatter(ScalarFormatter()); ax.minorticks_off()
        ax.tick_params(axis="x", labelrotation=90); ax.grid(alpha=0.3)
    axes[1, k].set_xlabel("scenes drawn (N)")
axes[0, 0].set_ylabel("centred RMSE (m)")
axes[1, 0].set_ylabel("bias (m)")
axes[0, 0].legend(frameon=False)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(f"{NB_DIR}/marea_vs_nidem_all_sites.{ext}", dpi=150, bbox_inches="tight")
print(f"-> {NB_DIR}/marea_vs_nidem_all_sites.png / .pdf")